In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs_v2/candidates/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("Retrieval_Repurchase") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

Mounted at /content/drive


In [ ]:
transactions = spark.read.parquet(INPUT_TRANS)
max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)
train_hist_start = val_start - datetime.timedelta(days=42)
test_hist_start = test_start - datetime.timedelta(days=42)

train_hist_df = transactions.filter((F.col("t_dat_date") >= train_hist_start) & (F.col("t_dat_date") < val_start))
test_hist_df = transactions.filter((F.col("t_dat_date") >= test_hist_start) & (F.col("t_dat_date") < test_start))

print(f"Train History (6W): {train_hist_start} -> {val_start}")
print(f"Train Target (1W):  {val_start} -> {test_start}")
print(f"Test History (6W):  {test_hist_start} -> {test_start}")
print(f"Test Target (1W):   {test_start} -> {max_date}")

Train History (6W): 2020-07-28 -> 2020-09-08
Train Target (1W):  2020-09-08 -> 2020-09-15
Test History (6W):  2020-08-04 -> 2020-09-15
Test Target (1W):   2020-09-15 -> 2020-09-22


In [ ]:
def generate_repurchase_candidates(history_df, top_n=15):
    user_item_latest = history_df.groupBy("customer_id", "article_id") \
        .agg(F.max("t_dat_date").alias("latest_buy_date"))

    window_spec = Window.partitionBy("customer_id").orderBy(F.col("latest_buy_date").desc())

    candidates = user_item_latest.withColumn("rn", F.row_number().over(window_spec)) \
        .filter(F.col("rn") <= top_n) \
        .select("customer_id", "article_id") \
        .withColumn("strategy", F.lit("repurchase"))
    return candidates

def evaluate_recall(candidates_df, target_df, target_start, target_end):
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(candidates_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print(f"Actuals: {total_actuals} | Hits: {hits} | Recall: {recall:.4f}")

In [ ]:
train_cands = generate_repurchase_candidates(train_hist_df, top_n=15)
train_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "train_repurchase.parquet")

test_cands = generate_repurchase_candidates(test_hist_df, top_n=15)
test_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "test_repurchase.parquet")

print("Evaluating TEST set:")
evaluate_recall(test_cands, transactions, test_start, max_date)

Evaluating TEST set:
Actuals: 207996 | Hits: 5218 | Recall: 0.0251
